# Bonus: RabbitMQ stream into the FreshCart lakehouse

Before running this notebook, run `FreshCart_ELT_Colab_Customized.ipynb` through its Snowflake setup and baseline pipeline cells in the **same Colab runtime**. This creates `/content/freshcart_elt`, its SQL files, and the Snowflake environment variables.

In [ ]:
!git clone https://github.com/AK-CODE-droid/freshcart-pipelines.git /content/freshcart-pipelines
%cd /content/freshcart-pipelines
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -y -qq rabbitmq-server
!service rabbitmq-server start

## Run the one-minute stream batch

The consumer starts first so its durable queue is bound before the two FreshCart order messages are published.

In [ ]:
%%bash
rm -f bonus_first.log bonus_replay.log
python -u stream/stream_to_lakehouse.py --elt-dir /content/freshcart_elt --seconds 60 --run-id bonus_stream_demo > bonus_first.log 2>&1 &
CONSUMER_PID=$!
sleep 5
python -u stream/freshcart_order_publisher.py
wait $CONSUMER_PID
cat bonus_first.log

## SCREENSHOT 1 - stream reaches gold

Take a screenshot of the previous cell's output. It must show `events_collected=2`, a positive `bronze_rows_loaded` value, and `gold_before` and `gold_after` totals.

## Replay the identical landing file

This deliberately processes the same raw file again. Snowflake's COPY history must return zero newly loaded bronze rows, and the gold totals must remain unchanged.

In [ ]:
%%bash
python -u stream/stream_to_lakehouse.py --elt-dir /content/freshcart_elt --run-id bonus_stream_demo --replay /content/freshcart_elt/landing/web/2026-09-30/web__bonus_stream_demo.json.gz > bonus_replay.log 2>&1
cat bonus_replay.log

## SCREENSHOT 2 - idempotent replay

Take a screenshot of the previous output. It must show `bronze_rows_loaded=0`, and identical `gold_before` and `gold_after` totals. In the report, explain that the stable landing filename is skipped by COPY history and the gold MERGE only updates a row when its `row_hash` changes.